# PKG Attrition — Source Profiling EDA

**Blocking profiling pass over the Neo4j staging transaction table and the deposit panel.**
Nothing downstream — no counterparty graph build, no episode table, no features — starts
until every stage here has produced a number.

Each stage is tagged with the open question it answers:

| Q | Question | Stage |
|---|---|---|
| Q1 | Staging table shape, grain, date range | 1 |
| — | Settlement lag (leakage surface) | 2 |
| Q2 | Rail composition, rows and dollars | 3 |
| Q4 | Direction convention consistency | 4 |
| Q3 | Counterparty name coverage by rail | 5 |
| Q5 | RTN / account coverage; hard-key construction | 5–6 |
| Q6 | Deposit grain, account types, accounts per customer | 7, 9 |
| Q9 | Account status field discovery | 8 |
| Q8 | Same-ledger test: do transactions reconcile to balance? | 10 |
| — | Join coverage: deposit book ↔ staging table | 11 |
| — | On-us self-payment ground-truth set | 12 |
| Q13 | Episode count under the current 30% rule | 13 |

**Run order matters.** Stage 0 resolves the schema and must be run first; it prints the
column mapping you paste back into `CAND_OVERRIDE`. Stages 1–6 profile the staging table
and can run independently of the deposit stages. Stage 10 needs both.

Every stage writes a CSV to `OUT_DIR`. Those CSVs are the deliverable, not the notebook
output — they are what gets quoted back in the reply to the brief.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================

CONFIG = dict(
    # --- sources -------------------------------------------------------------
    TXN_TABLE   = "<db>.<staging_transactions>",   # Neo4j ingestion staging table
    DEP_TABLE   = "<db>.<deposit_panel>",          # account-level deposit series
    ACCT_TABLE  = None,                            # optional account dimension; None = skip
    CUST_TABLE  = None,                            # optional customer dim (party_type, NAICS)

    # --- output --------------------------------------------------------------
    OUT_DIR     = "../eda/attrition",

    # --- scan control --------------------------------------------------------
    # Full-table scans are expensive. Every stage honours these. Start narrow,
    # widen once the schema is confirmed correct.
    DATE_MIN    = "2025-01-01",
    DATE_MAX    = "2025-12-31",
    FULL_RANGE_STAGES = {1, 2},   # stages that ignore DATE_MIN/MAX (cheap min/max only)

    # --- domain --------------------------------------------------------------
    # PNC's own routing numbers. An outbound leg carrying one of these is an
    # on-us transfer, NOT an external counterparty. Required for Stage 12.
    PNC_RTNS    = ["043000096"],   # <-- CONFIRM AND EXTEND. Legacy/acquired RTNs count.

    # --- thresholds ----------------------------------------------------------
    BALANCE_FLOOR       = 25_000,   # §6.2 small-balance exclusion in the brief
    RECON_TOL_REL       = 0.001,    # Stage 10: relative tolerance on ledger reconciliation
    RECON_SAMPLE_ACCTS  = 5_000,    # Stage 10: accounts sampled for reconciliation
    NAME_MIN_TOKENS     = 1,        # a "usable" name needs at least this many alpha tokens
    CURRENT_RULE_DROP   = 0.30,     # trailing-3 avg vs prior-6 avg, the existing rule
)

# Paste the Stage 0 output here on the second run. Anything left out of this dict
# falls back to the candidate search in CAND.
CAND_OVERRIDE = {
    # "txn_id":     "unique_txn_id",
    # "acct_id":    "acct_nbr",
    # "cpty_rtn":   "cntrpty_rtn_nbr",
}

# Candidate physical names per logical field, in priority order. Case-insensitive.
# Extend rather than replace — the resolver reports what it could not find.
CAND = {
    # --- transaction identity ------------------------------------------------
    "txn_id":     ["transaction_id", "txn_id", "trans_id", "event_id", "unique_txn_id", "tran_id"],
    "acct_id":    ["account_id", "acct_id", "acct_nbr", "account_number", "acct_num", "src_acct_nbr"],
    "cust_id":    ["cust_pwr_id", "customer_id", "mdm_id", "party_id", "cust_id"],

    # --- dates ---------------------------------------------------------------
    "txn_date":   ["transaction_dt", "txn_dt", "txn_date", "trans_dt", "effective_dt", "value_dt"],
    "post_date":  ["post_dt", "posting_dt", "posted_date", "settle_dt", "settlement_dt", "process_dt", "run_dt"],
    "txn_ts":     ["transaction_ts", "txn_ts", "event_ts", "trans_tmstmp", "txn_tmstmp"],

    # --- money ---------------------------------------------------------------
    "amount":     ["amount", "txn_amt", "tran_amt", "amt", "transaction_amount", "trans_amount"],
    "direction":  ["direction", "dr_cr_cd", "debit_credit_ind", "dr_cr_ind", "credit_debit", "dc_ind", "flow_dir"],
    "rail":       ["rail", "payment_rail", "channel", "txn_type", "transaction_type", "tran_cd", "product_cd"],

    # --- counterparty --------------------------------------------------------
    "cpty_name":  ["cpty_name", "counterparty_name", "cntrpty_nm", "beneficiary_name", "bene_name",
                   "originator_name", "orig_name", "other_party_name", "payee_name"],
    "cpty_acct":  ["cpty_acct", "counterparty_account", "cntrpty_acct_nbr", "bene_acct_nbr",
                   "other_party_acct", "dest_acct_nbr"],
    "cpty_rtn":   ["cpty_rtn", "counterparty_rtn", "cntrpty_rtn_nbr", "bene_rtn", "routing_number",
                   "aba_nbr", "rtn", "other_party_rtn"],
    "cpty_bank":  ["cpty_bank_name", "bene_bank_name", "counterparty_bank", "fi_name"],

    # --- deposit panel -------------------------------------------------------
    "dep_acct":   ["account_id", "acct_id", "acct_nbr", "account_number", "acct_num"],
    "dep_cust":   ["cust_pwr_id", "customer_id", "mdm_id", "party_id", "cust_id"],
    "dep_date":   ["as_of_dt", "business_dt", "balance_dt", "snapshot_dt", "eff_dt", "dt", "month_end_dt"],
    "dep_bal":    ["balance", "ledger_bal", "ledger_balance", "avg_balance", "eod_balance",
                   "closing_balance", "collected_bal", "bal_amt"],
    "dep_type":   ["account_type", "acct_type_cd", "product_type", "prod_cd", "acct_prod_cd"],
    "dep_status": ["account_status", "acct_status_cd", "status_cd", "acct_stat", "open_closed_ind"],
    "dep_open":   ["open_dt", "acct_open_dt", "account_open_date"],
    "dep_close":  ["close_dt", "acct_close_dt", "account_close_date", "closed_dt"],
}

# Tokens that mark a name field as junk rather than a real counterparty name.
# Extend from what Stage 5 shows in the top-value listing.
NAME_JUNK = [
    "UNKNOWN", "N/A", "NA", "NONE", "NULL", "UNAVAILABLE", "NOT AVAILABLE",
    "SEE MEMO", "DESCRIPTION", "MISC", "MISCELLANEOUS", "XXXX", "*", "-",
    "DEPOSIT", "WITHDRAWAL", "TRANSFER", "PAYMENT",
]

In [ ]:
# ============================================================================
# IMPORTS & HELPERS
# ============================================================================
import os
import json
import pandas as pd

from pyspark.sql import SparkSession, functions as F, Window as W, types as T

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "400")

OUT = CONFIG["OUT_DIR"]
os.makedirs(OUT, exist_ok=True)

_RESULTS = {}   # stage -> headline dict, assembled into the Stage 14 summary


def save(pdf: pd.DataFrame, name: str, note: str = "") -> pd.DataFrame:
    """Write a small result frame to OUT_DIR and echo it."""
    path = os.path.join(OUT, f"{name}.csv")
    pdf.to_csv(path, index=False)
    print(f"\n--- {name} {'· ' + note if note else ''}")
    with pd.option_context("display.max_rows", 60, "display.width", 200):
        print(pdf.to_string(index=False))
    print(f"    -> {path}")
    return pdf


def resolve(df, logical: str, required: bool = True):
    """Physical column name for a logical field. CAND_OVERRIDE wins."""
    if logical in CAND_OVERRIDE:
        return CAND_OVERRIDE[logical]
    lower = {c.lower(): c for c in df.columns}
    for cand in CAND.get(logical, []):
        if cand.lower() in lower:
            return lower[cand.lower()]
    if required:
        raise KeyError(
            f"Could not resolve '{logical}'. Columns available: {sorted(df.columns)}\n"
            f"Add the physical name to CAND_OVERRIDE."
        )
    return None


def date_filter(df, datecol, stage):
    """Apply the scan window unless this stage is exempt."""
    if stage in CONFIG["FULL_RANGE_STAGES"]:
        return df
    return df.filter(
        (F.col(datecol) >= F.lit(CONFIG["DATE_MIN"])) &
        (F.col(datecol) <= F.lit(CONFIG["DATE_MAX"]))
    )


def share_table(df, group_cols, amount_col, name, note=""):
    """Rows and dollars by group, with shares. The workhorse for Stages 3 and 5.

    Dollars and rows are reported side by side deliberately: a rail can be 40% of
    rows and 2% of dollars (card) or the reverse (wire). A coverage figure quoted
    on rows alone is not a statement about value at risk.
    """
    g = (df.groupBy(*group_cols)
           .agg(F.count(F.lit(1)).alias("n_rows"),
                F.sum(F.abs(F.col(amount_col))).alias("abs_dollars"))
           .toPandas())
    g["share_rows"] = g["n_rows"] / g["n_rows"].sum()
    g["share_dollars"] = g["abs_dollars"] / g["abs_dollars"].sum()
    g = g.sort_values("abs_dollars", ascending=False)
    return save(g, name, note)


def norm_name(col):
    """Uppercase, strip punctuation and legal suffixes, collapse whitespace.

    Deliberately NOT fuzzy. Exact match on this normalisation is the v1 matcher;
    fuzzy expansion is a later, separately-calibrated layer.
    """
    c = F.upper(F.trim(col))
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(LLC|L L C|INC|INCORPORATED|CORP|CORPORATION|CO|LP|LLP|"
                            r"PLLC|PC|LTD|LIMITED|TRUST|DBA|THE)\b", " ")
    c = F.regexp_replace(c, r"\s+", " ")
    return F.trim(c)


def norm_acct(col):
    """Account-number normalisation: alnum only, leading zeros stripped, uppercased.

    Leading zeros are the same trap as zip_cd — an account arriving from one rail
    zero-padded and from another not is two keys for one account.
    """
    c = F.upper(F.regexp_replace(F.coalesce(col, F.lit("")), r"[^A-Za-z0-9]", ""))
    return F.regexp_replace(c, r"^0+", "")


def rtn_valid(col):
    """ABA checksum: 3(d1+d4+d7) + 7(d2+d5+d8) + (d3+d6+d9) ≡ 0 mod 10.

    A syntactically valid RTN is not a real one, but an invalid RTN is definitely
    junk — and junk RTNs silently fragment the counterparty key.
    """
    d = [F.substring(col, i, 1).cast("int") for i in range(1, 10)]
    tot = (3 * (d[0] + d[3] + d[6])) + (7 * (d[1] + d[4] + d[7])) + (d[2] + d[5] + d[8])
    return (F.length(col) == 9) & col.rlike(r"^[0-9]{9}$") & ((tot % 10) == 0)


def name_usable(col):
    """A name is usable if it survives normalisation, is not a junk token, and
    carries at least NAME_MIN_TOKENS alphabetic tokens."""
    n = norm_name(col)
    junk = F.upper(F.trim(F.coalesce(col, F.lit("")))).isin([j.upper() for j in NAME_JUNK])
    n_tok = F.size(F.split(n, " "))
    return (n != "") & (~junk) & (n_tok >= CONFIG["NAME_MIN_TOKENS"])


print("helpers loaded ·", OUT)

---
## Stage 0 — Schema discovery

**Run this first, alone.** It resolves every logical field against the physical
schema and prints what it could not find. Paste the corrections into
`CAND_OVERRIDE` and re-run before touching any other stage.

A wrong column mapping here does not error — it produces a plausible-looking
profile of the wrong field. That failure mode is the whole reason this stage exists.

In [ ]:
txn_raw = spark.table(CONFIG["TXN_TABLE"])
dep_raw = spark.table(CONFIG["DEP_TABLE"])

rows = []
for tbl_name, df, fields in [
    ("txn", txn_raw, ["txn_id", "acct_id", "cust_id", "txn_date", "post_date", "txn_ts",
                      "amount", "direction", "rail", "cpty_name", "cpty_acct",
                      "cpty_rtn", "cpty_bank"]),
    ("dep", dep_raw, ["dep_acct", "dep_cust", "dep_date", "dep_bal", "dep_type",
                      "dep_status", "dep_open", "dep_close"]),
]:
    dtypes = dict(df.dtypes)
    for f in fields:
        phys = resolve(df, f, required=False)
        rows.append({
            "table": tbl_name,
            "logical": f,
            "physical": phys or "*** NOT FOUND ***",
            "dtype": dtypes.get(phys, ""),
            "source": "override" if f in CAND_OVERRIDE else ("candidate" if phys else "unresolved"),
        })

schema_map = save(pd.DataFrame(rows), "00_schema_map", "resolve everything before proceeding")

print("\n--- txn table: FULL column list")
print(pd.DataFrame(txn_raw.dtypes, columns=["column", "dtype"]).to_string(index=False))
print("\n--- dep table: FULL column list")
print(pd.DataFrame(dep_raw.dtypes, columns=["column", "dtype"]).to_string(index=False))

# Unmapped columns often hold the answer to Q9 (status) and to the rail taxonomy.
mapped = {r["physical"] for r in rows}
print("\n--- txn columns NOT mapped to any logical field (scan these for status/rail/flags):")
print(sorted(set(txn_raw.columns) - mapped))
print("\n--- dep columns NOT mapped:")
print(sorted(set(dep_raw.columns) - mapped))

unresolved = [r["logical"] for r in rows if r["source"] == "unresolved"]
if unresolved:
    print(f"\n*** UNRESOLVED: {unresolved} — populate CAND_OVERRIDE and re-run Stage 0. ***")

In [ ]:
# Bind resolved names once. Everything below uses these.
C = {f: resolve(txn_raw, f, required=False) for f in
     ["txn_id", "acct_id", "cust_id", "txn_date", "post_date", "txn_ts",
      "amount", "direction", "rail", "cpty_name", "cpty_acct", "cpty_rtn", "cpty_bank"]}
D = {f: resolve(dep_raw, f, required=False) for f in
     ["dep_acct", "dep_cust", "dep_date", "dep_bal", "dep_type",
      "dep_status", "dep_open", "dep_close"]}
print(json.dumps({"txn": C, "dep": D}, indent=2))

---
## Stage 1 — Staging table shape and grain  *(Q1)*

Row count, date range, distinct customers and accounts, and whether the
transaction id is actually unique. If it is not, every "per transaction"
figure below is a per-leg figure and the double-entry structure has to be
handled explicitly.

In [ ]:
STAGE = 1
txn = txn_raw

shape = txn.agg(
    F.count(F.lit(1)).alias("n_rows"),
    F.countDistinct(F.col(C["txn_id"])).alias("n_txn_ids") if C["txn_id"] else F.lit(None).alias("n_txn_ids"),
    F.countDistinct(F.col(C["acct_id"])).alias("n_accounts"),
    F.countDistinct(F.col(C["cust_id"])).alias("n_customers") if C["cust_id"] else F.lit(None).alias("n_customers"),
    F.min(F.col(C["txn_date"])).alias("min_txn_date"),
    F.max(F.col(C["txn_date"])).alias("max_txn_date"),
    F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"),
).toPandas()

shape["rows_per_txn_id"] = (shape["n_rows"] / shape["n_txn_ids"]) if C["txn_id"] else None
save(shape.T.reset_index().rename(columns={"index": "metric", 0: "value"}),
     "01_staging_shape", "rows_per_txn_id > 1 means double-entry legs, not duplicates")

_RESULTS["shape"] = shape.iloc[0].to_dict()

In [ ]:
# Monthly volume — establishes whether coverage is uniform or whether early
# months are partial. A ramp at the start of the range is an ingestion artefact,
# not a business trend, and it will masquerade as a level shift in every feature.
STAGE = 1
monthly = (txn
    .withColumn("month", F.date_format(F.col(C["txn_date"]), "yyyy-MM"))
    .groupBy("month")
    .agg(F.count(F.lit(1)).alias("n_rows"),
         F.countDistinct(F.col(C["acct_id"])).alias("n_accounts"),
         F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"))
    .orderBy("month")
    .toPandas())
save(monthly, "01_monthly_volume", "check for ingestion ramp at the boundaries")

In [ ]:
# Nullity across every column, on a sample. Cheap, and it catches columns that
# are present in the schema but never populated — which is the most common way a
# field gets planned into a feature and then found empty three weeks later.
STAGE = 1
samp = txn.sample(False, 0.01, seed=42).cache()
n_samp = samp.count()

nullity = samp.select([
    (F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)) / F.lit(n_samp))
    .alias(c) for c in txn.columns
]).toPandas().T.reset_index()
nullity.columns = ["column", "null_or_blank_share"]
save(nullity.sort_values("null_or_blank_share", ascending=False),
     "01_column_nullity", f"1% sample, n={n_samp:,}")

---
## Stage 2 — Dates and settlement lag

The standing concern from the manifest: **settlement lag is a leakage surface.**
If the only date on the row is the settlement date, then a transaction that
occurred on day *t* is not observable until *t+k*, and any daily-grain sequence
claim ("receivables moved before payables") is measuring the settlement pipeline
rather than customer behaviour.

This stage decides whether daily grain is real or an illusion. Three outcomes:

- **Two dates present, lag profiled** → daily grain is usable with a documented
  as-of rule (features at *t* may only use rows where `post_date ≤ t`).
- **One date, and it is the settlement date** → the effective grain is
  `daily minus the lag tail`. Sequence claims need a lag-aware buffer.
- **One date, and it is the transaction date with no posting date** → the panel
  cannot be reconstructed as-of; it is retrospective only. That is a hard
  constraint on any real-time scoring design.

In [ ]:
STAGE = 2
if C["post_date"] and C["txn_date"] and C["post_date"] != C["txn_date"]:
    lag = (txn
        .withColumn("lag_days", F.datediff(F.col(C["post_date"]), F.col(C["txn_date"])))
        .filter(F.col("lag_days").isNotNull()))

    by_rail = (lag.groupBy(C["rail"] if C["rail"] else F.lit("ALL"))
        .agg(F.count(F.lit(1)).alias("n"),
             F.mean("lag_days").alias("mean_lag"),
             F.expr("percentile_approx(lag_days, 0.5)").alias("p50_lag"),
             F.expr("percentile_approx(lag_days, 0.9)").alias("p90_lag"),
             F.expr("percentile_approx(lag_days, 0.99)").alias("p99_lag"),
             F.max("lag_days").alias("max_lag"),
             F.sum(F.when(F.col("lag_days") < 0, 1).otherwise(0)).alias("n_negative"))
        .toPandas())
    save(by_rail, "02_settlement_lag_by_rail",
         "p99_lag is the as-of buffer; n_negative > 0 means the date semantics are not what we think")
else:
    print("*** Only one usable date column. Daily grain is settlement-dated. ***")
    print("*** Record this as a stated limit; sequence claims need a lag buffer. ***")
    save(pd.DataFrame([{"finding": "single_date_column",
                        "txn_date": C["txn_date"], "post_date": C["post_date"]}]),
         "02_settlement_lag_by_rail")

In [ ]:
# Intraday timestamp availability. Sub-daily is only worth pursuing if the
# timestamp is a real event time rather than a batch-load time — a giveaway is
# mass concentration at midnight or at a handful of batch windows.
STAGE = 2
if C["txn_ts"]:
    hours = (txn.withColumn("hour", F.hour(F.col(C["txn_ts"])))
                .groupBy("hour").count().orderBy("hour").toPandas())
    save(hours, "02_hour_of_day",
         "mass at 00:00 or a few spikes = batch load time, not event time; sub-daily is not real")
else:
    print("No timestamp column resolved — daily is the floor.")

---
## Stage 3 — Rail composition  *(Q2)*

Rails and dollars. Two things come out of this: the rail taxonomy that the rail
metrics spec has been blocked on, and the denominator for Stage 5's coverage
read. Internal book transfers are called out separately — they are movements
between two PNC accounts and are neither a payment nor an off-us flow, so
leaving them in inflates every share.

In [ ]:
STAGE = 3
txn_w = date_filter(txn, C["txn_date"], STAGE).cache()

share_table(txn_w, [C["rail"]], C["amount"], "03_rail_composition",
            "this is the rail taxonomy — reconcile against PKG_RAIL_METRICS_SPEC")

In [ ]:
# Rail × direction. Some rails are one-directional by construction (an originated
# wire vs a received wire may be distinct type codes rather than one code plus a
# direction flag). If so, the direction column is redundant on those rails and
# informative only on others.
STAGE = 3
if C["direction"]:
    share_table(txn_w, [C["rail"], C["direction"]], C["amount"], "03_rail_x_direction")

In [ ]:
# Amount distribution per rail. Card and ATM sit at a different order of magnitude
# from wire; a single dollar threshold applied across rails is meaningless.
STAGE = 3
amt_dist = (txn_w.groupBy(C["rail"]).agg(
        F.count(F.lit(1)).alias("n"),
        F.expr(f"percentile_approx(abs({C['amount']}), 0.05)").alias("p05"),
        F.expr(f"percentile_approx(abs({C['amount']}), 0.50)").alias("p50"),
        F.expr(f"percentile_approx(abs({C['amount']}), 0.95)").alias("p95"),
        F.expr(f"percentile_approx(abs({C['amount']}), 0.999)").alias("p999"),
        F.max(F.abs(F.col(C["amount"]))).alias("max"),
        F.sum(F.when(F.col(C["amount"]) == 0, 1).otherwise(0)).alias("n_zero_amount"),
    ).toPandas())
save(amt_dist, "03_amount_by_rail", "n_zero_amount > 0 needs an explanation before any weighting")

---
## Stage 4 — Direction convention  *(Q4)*

Stated as trustworthy, so this is a verification rather than an investigation.
The failure mode being checked: sign convention and direction flag disagreeing
on some rails, which produces a customer whose outflow is booked as inflow on
one rail only. That does not show up as an error anywhere — it shows up as a
customer with an implausible net flow, eighteen months later.

In [ ]:
STAGE = 4
if C["direction"]:
    conv = (txn_w.groupBy(C["direction"])
        .agg(F.count(F.lit(1)).alias("n"),
             F.sum(F.when(F.col(C["amount"]) > 0, 1).otherwise(0)).alias("n_amt_positive"),
             F.sum(F.when(F.col(C["amount"]) < 0, 1).otherwise(0)).alias("n_amt_negative"),
             F.sum(F.when(F.col(C["amount"]) == 0, 1).otherwise(0)).alias("n_amt_zero"))
        .toPandas())
    conv["pct_positive"] = conv["n_amt_positive"] / conv["n"]
    save(conv, "04_direction_sign_convention",
         "each direction value should be ~100% one sign, or the sign is carrying no information")

    # Same check within rail — this is where the disagreement hides.
    conv_rail = (txn_w.groupBy(C["rail"], C["direction"])
        .agg(F.count(F.lit(1)).alias("n"),
             F.mean(F.when(F.col(C["amount"]) > 0, 1.0).otherwise(0.0)).alias("pct_positive"))
        .toPandas())
    save(conv_rail.sort_values(["pct_positive"]), "04_direction_by_rail",
         "any (rail, direction) cell with pct_positive between 0.05 and 0.95 is a mixed convention")

---
## Stage 5 — Counterparty identifiability  *(Q3, Q5)*

**The stage that decides whether Group A exists.**

Three identifiers, profiled separately and jointly, by rows *and* by dollars,
split by rail:

- **name** — needed for same-name self-payment detection and FI classification
- **RTN** — needed to know which institution the money went to
- **account** — needed for the hard counterparty key

The joint table is the important one. `RTN + account` is a hard key and gives a
stable counterparty node without any name matching at all. Name-only rows fall
back to fuzzy resolution with all its failure modes. Rows with neither are
invisible to counterparty analysis regardless of how much dollar value they carry.

**Expect this to be strongly rail-dependent.** Rail mix correlates with industry
and size, so a coverage figure quoted in aggregate is a biased statement about
which customers are analysable.

In [ ]:
STAGE = 5
cp = (txn_w
      .withColumn("_has_name", name_usable(F.col(C["cpty_name"])) if C["cpty_name"] else F.lit(False))
      .withColumn("_rtn_clean", F.regexp_replace(F.coalesce(F.col(C["cpty_rtn"]).cast("string"), F.lit("")), r"[^0-9]", "")
                  if C["cpty_rtn"] else F.lit(""))
      .withColumn("_acct_clean", norm_acct(F.col(C["cpty_acct"]).cast("string")) if C["cpty_acct"] else F.lit(""))
     )
# Zero-pad an RTN that lost leading zeros to an integer cast upstream — the same
# failure as zip_cd. A 8-digit RTN is almost always a 9-digit one missing a zero.
cp = cp.withColumn("_rtn_clean",
                   F.when(F.length("_rtn_clean").between(1, 8), F.lpad("_rtn_clean", 9, "0"))
                    .otherwise(F.col("_rtn_clean")))
cp = (cp
      .withColumn("_has_rtn", (F.length("_rtn_clean") == 9))
      .withColumn("_rtn_checksum_ok", rtn_valid(F.col("_rtn_clean")))
      .withColumn("_has_acct", F.length("_acct_clean") >= 4)
      .cache())

cov = (cp.groupBy(C["rail"]).agg(
        F.count(F.lit(1)).alias("n_rows"),
        F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"),
        F.mean(F.col("_has_name").cast("double")).alias("name_rate_rows"),
        F.mean(F.col("_has_rtn").cast("double")).alias("rtn_rate_rows"),
        F.mean(F.col("_rtn_checksum_ok").cast("double")).alias("rtn_valid_rate_rows"),
        F.mean(F.col("_has_acct").cast("double")).alias("acct_rate_rows"),
        (F.sum(F.when(F.col("_has_name"), F.abs(F.col(C["amount"]))).otherwise(0.0))
         / F.sum(F.abs(F.col(C["amount"])))).alias("name_rate_dollars"),
        (F.sum(F.when(F.col("_has_rtn") & F.col("_has_acct"), F.abs(F.col(C["amount"]))).otherwise(0.0))
         / F.sum(F.abs(F.col(C["amount"])))).alias("hardkey_rate_dollars"),
    ).toPandas())
cov["share_dollars"] = cov["abs_dollars"] / cov["abs_dollars"].sum()
save(cov.sort_values("abs_dollars", ascending=False), "05_cpty_coverage_by_rail",
     "*** THE Q3 ANSWER. name_rate_dollars is what gates Group A. ***")

_RESULTS["name_rate_dollars_overall"] = float(
    (cov["name_rate_dollars"] * cov["abs_dollars"]).sum() / cov["abs_dollars"].sum())
_RESULTS["hardkey_rate_dollars_overall"] = float(
    (cov["hardkey_rate_dollars"] * cov["abs_dollars"]).sum() / cov["abs_dollars"].sum())

In [ ]:
# Joint identifiability classes. This is the table that determines the
# counterparty-resolution architecture: if hard-key dominates, name matching is a
# fallback layer and its error rate barely matters. If name-only dominates, the
# matcher's calibration is on the critical path.
STAGE = 5
cp2 = cp.withColumn("id_class",
    F.when(F.col("_has_rtn") & F.col("_has_acct") & F.col("_has_name"), "hardkey+name")
     .when(F.col("_has_rtn") & F.col("_has_acct"), "hardkey_only")
     .when(F.col("_has_name") & F.col("_has_rtn"), "name+rtn")
     .when(F.col("_has_name"), "name_only")
     .when(F.col("_has_rtn"), "rtn_only")
     .otherwise("unidentifiable"))

share_table(cp2, ["id_class"], C["amount"], "05_identifiability_classes",
            "*** the architecture decision: hardkey share vs name-only share ***")
share_table(cp2, [C["rail"], "id_class"], C["amount"], "05_identifiability_by_rail")

In [ ]:
# What the unusable names actually look like. Feeds the NAME_JUNK list — the
# top-50 by frequency will contain the sentinels this schema happens to use, and
# they are never the ones you guessed.
STAGE = 5
if C["cpty_name"]:
    junk = (cp.filter(~F.col("_has_name"))
              .groupBy(F.upper(F.trim(F.col(C["cpty_name"]))).alias("raw_name"))
              .agg(F.count(F.lit(1)).alias("n"),
                   F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"))
              .orderBy(F.desc("n")).limit(50).toPandas())
    save(junk, "05_unusable_name_values", "extend NAME_JUNK from this list and re-run")

    # And the top real names — these are the hubs. Payroll processors, card
    # networks and the bank's own book-transfer accounts should be visible here,
    # and each needs a different handling policy (signal hub / noise hub / internal).
    top = (cp.filter(F.col("_has_name"))
             .groupBy(norm_name(F.col(C["cpty_name"])).alias("norm_name"))
             .agg(F.count(F.lit(1)).alias("n"),
                  F.countDistinct(F.col(C["acct_id"])).alias("n_customers"),
                  F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"))
             .orderBy(F.desc("n_customers")).limit(100).toPandas())
    save(top, "05_top_counterparty_names",
         "seed for the hub taxonomy: payroll / card network / internal / genuine anchor")

In [ ]:
# Top destination RTNs. This is the FI list the brief asks for in §12 — derived
# rather than assembled by hand, and weighted by where the money actually goes.
STAGE = 5
if C["cpty_rtn"]:
    top_rtn = (cp.filter(F.col("_has_rtn"))
        .groupBy("_rtn_clean")
        .agg(F.count(F.lit(1)).alias("n"),
             F.countDistinct(F.col(C["acct_id"])).alias("n_customers"),
             F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"),
             F.first(F.col(C["cpty_bank"]), ignorenulls=True).alias("bank_name_sample")
             if C["cpty_bank"] else F.lit(None).alias("bank_name_sample"))
        .orderBy(F.desc("n_customers")).limit(200).toPandas())
    top_rtn["is_pnc"] = top_rtn["_rtn_clean"].isin(CONFIG["PNC_RTNS"])
    save(top_rtn, "05_top_destination_rtns",
         "join to FDIC/NCUA for the FI registry; is_pnc rows are on-us, not external")

---
## Stage 6 — Counterparty key and fan-out

Builds the hard key and sizes the resulting graph. The fan-out number decides
whether the deposit-anchored ego extraction is a Spark job or a Spark problem:
distinct counterparties per customer per month, times the corporate book, is the
edge count of the Tier-1 monthly graph.

In [ ]:
STAGE = 6
keyed = (cp
    .withColumn("cpty_key",
        F.when(F.col("_has_rtn") & F.col("_has_acct"),
               F.sha2(F.concat_ws("|", F.col("_rtn_clean"), F.col("_acct_clean")), 256))
         .when(F.col("_has_name"),
               F.concat(F.lit("NM:"), F.sha2(norm_name(F.col(C["cpty_name"])), 256)))
         .otherwise(F.lit(None)))
    .withColumn("cpty_key_type",
        F.when(F.col("_has_rtn") & F.col("_has_acct"), "hard")
         .when(F.col("_has_name"), "name").otherwise("none"))
    .withColumn("month", F.date_format(F.col(C["txn_date"]), "yyyy-MM")))

fanout = (keyed.filter(F.col("cpty_key").isNotNull())
    .groupBy(C["acct_id"], "month")
    .agg(F.countDistinct("cpty_key").alias("n_cpty"),
         F.count(F.lit(1)).alias("n_txn"))
    .groupBy("month")
    .agg(F.count(F.lit(1)).alias("n_account_months"),
         F.mean("n_cpty").alias("mean_cpty"),
         F.expr("percentile_approx(n_cpty, 0.5)").alias("p50_cpty"),
         F.expr("percentile_approx(n_cpty, 0.95)").alias("p95_cpty"),
         F.expr("percentile_approx(n_cpty, 0.999)").alias("p999_cpty"),
         F.max("n_cpty").alias("max_cpty"),
         F.sum("n_cpty").alias("total_edges"))
    .orderBy("month").toPandas())
save(fanout, "06_cpty_fanout_by_month",
     "total_edges is the Tier-1 monthly edge count; p999/max sizes the hub tail")

In [ ]:
# Reverse fan-in: how many distinct PNC customers touch each counterparty. This is
# what makes lost_cpty_health_index computable — a counterparty seen by only one
# customer has no independent health signal, so the discriminator is undefined for it.
STAGE = 6
fanin = (keyed.filter(F.col("cpty_key").isNotNull())
    .groupBy("cpty_key")
    .agg(F.countDistinct(F.col(C["acct_id"])).alias("n_customers"))
    .groupBy("n_customers").count().orderBy("n_customers").limit(50).toPandas())
save(fanin, "06_cpty_fanin_distribution",
     "n_customers = 1 counterparties cannot support lost_cpty_health_index")

shared = (keyed.filter(F.col("cpty_key").isNotNull())
    .groupBy("cpty_key").agg(F.countDistinct(F.col(C["acct_id"])).alias("n_customers"))
    .agg(F.count(F.lit(1)).alias("n_cpty_total"),
         F.sum(F.when(F.col("n_customers") >= 2, 1).otherwise(0)).alias("n_cpty_shared"),
         F.sum(F.when(F.col("n_customers") >= 5, 1).otherwise(0)).alias("n_cpty_5plus"))
    .toPandas())
shared["share_shared"] = shared["n_cpty_shared"] / shared["n_cpty_total"]
save(shared, "06_cpty_shared_summary", "share_shared bounds Group C coverage")

---
## Stage 7 — Deposit panel grain  *(Q6)*

Grain, account types, and the balance field's semantics. The specific thing
being tested: is there one row per account per day, and does `balance` mean
end-of-day ledger or a period average. Those two read identically in a column
name and behave completely differently under a 30% rule.

In [ ]:
STAGE = 7
dep = date_filter(dep_raw, D["dep_date"], STAGE).cache()

dshape = dep.agg(
    F.count(F.lit(1)).alias("n_rows"),
    F.countDistinct(F.col(D["dep_acct"])).alias("n_accounts"),
    F.countDistinct(F.col(D["dep_cust"])).alias("n_customers") if D["dep_cust"] else F.lit(None).alias("n_customers"),
    F.countDistinct(F.col(D["dep_date"])).alias("n_dates"),
    F.min(F.col(D["dep_date"])).alias("min_date"),
    F.max(F.col(D["dep_date"])).alias("max_date"),
).toPandas()
dshape["rows_per_acct_date"] = dshape["n_rows"] / (dshape["n_accounts"] * dshape["n_dates"])
save(dshape.T.reset_index().rename(columns={"index": "metric", 0: "value"}), "07_deposit_shape")

In [ ]:
# Are the dates every calendar day, business days only, or month-ends?
STAGE = 7
dates = (dep.select(F.col(D["dep_date"]).alias("d")).distinct()
            .withColumn("dow", F.date_format("d", "E"))
            .withColumn("is_month_end", F.col("d") == F.last_day("d"))
            .groupBy("dow", "is_month_end").count().toPandas())
save(dates, "07_deposit_date_cadence",
     "*** THE Q6/Q7 ANSWER: month-end only, business days, or every day ***")

In [ ]:
# Duplicate grain check — more than one row per (account, date) means there is a
# hidden dimension (balance type, currency, sub-ledger) that has to be picked
# deliberately rather than by whichever row sorts first.
STAGE = 7
dupes = (dep.groupBy(D["dep_acct"], D["dep_date"]).count()
            .filter(F.col("count") > 1).limit(20).toPandas())
if len(dupes):
    print("*** MULTIPLE ROWS PER (account, date) — find the hidden dimension before aggregating ***")
save(dupes, "07_deposit_grain_duplicates")

In [ ]:
# Account types, with balance scale. Corporate DDA/MMDA is the study population;
# everything else needs an explicit include/exclude decision.
STAGE = 7
if D["dep_type"]:
    types = (dep.groupBy(D["dep_type"])
        .agg(F.count(F.lit(1)).alias("n_rows"),
             F.countDistinct(F.col(D["dep_acct"])).alias("n_accounts"),
             F.expr(f"percentile_approx({D['dep_bal']}, 0.5)").alias("p50_balance"),
             F.mean(F.col(D["dep_bal"])).alias("mean_balance"),
             F.sum(F.when(F.col(D["dep_bal"]) < 0, 1).otherwise(0)).alias("n_negative"))
        .orderBy(F.desc("n_accounts")).toPandas())
    save(types, "07_account_types", "identify the corporate DDA/MMDA codes here")

---
## Stage 8 — Account status discovery  *(Q9)*

**The highest-value output in this notebook.** A real closure/dormancy flag
breaks the circularity in the current design: right now T0 is defined off the
balance series, so a deposit model predicts a deposit-derived event and wins by
construction. A status-based label is an independent target and makes the graph's
contribution measurable rather than structurally handicapped.

This stage looks for the field three ways: a named status column, open/close
dates, and any unmapped column whose values look categorical and status-like.

In [ ]:
STAGE = 8
if D["dep_status"]:
    st = (dep.groupBy(D["dep_status"])
            .agg(F.count(F.lit(1)).alias("n_rows"),
                 F.countDistinct(F.col(D["dep_acct"])).alias("n_accounts"))
            .orderBy(F.desc("n_rows")).toPandas())
    save(st, "08_account_status_values", "*** Q9: the label source, if these values are real ***")

    # Does status actually transition, or is it a static current-state field
    # stamped on every historical row? A current-state field cannot date a closure
    # and is unusable as a time-varying label — it would leak the outcome into
    # every prior month.
    trans = (dep.groupBy(D["dep_acct"])
                .agg(F.countDistinct(F.col(D["dep_status"])).alias("n_distinct_status"))
                .groupBy("n_distinct_status").count().toPandas())
    save(trans, "08_status_transitions",
         "*** if n_distinct_status is 1 for ~every account, the field is CURRENT-STATE and leaks ***")
else:
    print("No status column resolved. Checking unmapped columns for status-like values...")

In [ ]:
# Open / close dates as an alternative label source.
STAGE = 8
if D["dep_close"]:
    closes = (dep.select(D["dep_acct"], D["dep_close"]).distinct()
        .withColumn("closed", F.col(D["dep_close"]).isNotNull())
        .groupBy("closed").agg(F.countDistinct(F.col(D["dep_acct"])).alias("n_accounts"))
        .toPandas())
    save(closes, "08_close_date_coverage", "a populated close_dt is a dated label, which is what we want")

    by_month = (dep.filter(F.col(D["dep_close"]).isNotNull())
        .select(D["dep_acct"], D["dep_close"]).distinct()
        .withColumn("close_month", F.date_format(F.col(D["dep_close"]), "yyyy-MM"))
        .groupBy("close_month").count().orderBy("close_month").toPandas())
    save(by_month, "08_closures_by_month", "*** Q13 upper bound: the true attrition event count ***")
    _RESULTS["closures_in_window"] = int(by_month["count"].sum()) if len(by_month) else 0

In [ ]:
# Brute-force search of unmapped columns for a status field: low cardinality,
# string-ish, and containing a recognisable status token.
STAGE = 8
STATUS_TOKENS = ["CLOS", "OPEN", "ACTIVE", "DORMANT", "INACTIVE", "SUSPEND", "FROZEN",
                 "TERMINAT", "A", "C", "D", "I"]
cands = []
samp_d = dep.sample(False, 0.02, seed=7).cache()
for c, dt in dep.dtypes:
    if c in D.values():
        continue
    if dt not in ("string", "int", "bigint", "smallint", "tinyint"):
        continue
    try:
        vals = (samp_d.select(F.upper(F.col(c).cast("string")).alias("v"))
                      .filter(F.col("v").isNotNull())
                      .groupBy("v").count().orderBy(F.desc("count")).limit(10).toPandas())
    except Exception:
        continue
    if 1 < len(vals) <= 10:
        hit = any(any(t in str(v) for t in STATUS_TOKENS) for v in vals["v"])
        cands.append({"column": c, "dtype": dt, "n_values": len(vals),
                      "top_values": "|".join(str(v) for v in vals["v"].tolist()),
                      "status_token_hit": hit})
save(pd.DataFrame(cands).sort_values("status_token_hit", ascending=False)
     if cands else pd.DataFrame([{"finding": "no low-cardinality candidates"}]),
     "08_status_column_candidates", "*** read this row by row — Q9 is probably in here ***")

---
## Stage 9 — Customer ↔ account roll-up  *(Q6)*

One customer, several accounts, possibly of different types. The roll-up rule
is a definitional choice with consequences: a client who closes one account and
keeps two others is not attriting, but an account-level label will say they are.
This stage measures how often that shape occurs.

In [ ]:
STAGE = 9
if D["dep_cust"]:
    per_cust = (dep.select(D["dep_cust"], D["dep_acct"]).distinct()
        .groupBy(D["dep_cust"]).agg(F.count(F.lit(1)).alias("n_accounts")))
    dist = per_cust.groupBy("n_accounts").count().orderBy("n_accounts").limit(50).toPandas()
    save(dist, "09_accounts_per_customer", "the multi-account tail is where the label gets ambiguous")

    if D["dep_type"]:
        mix = (dep.select(D["dep_cust"], D["dep_acct"], D["dep_type"]).distinct()
            .groupBy(D["dep_cust"]).agg(F.countDistinct(F.col(D["dep_type"])).alias("n_types"))
            .groupBy("n_types").count().orderBy("n_types").toPandas())
        save(mix, "09_account_types_per_customer")

    # The specific shape that breaks an account-level label: at least one account
    # closed while at least one other stayed open.
    if D["dep_close"]:
        partial = (dep.select(D["dep_cust"], D["dep_acct"], D["dep_close"]).distinct()
            .groupBy(D["dep_cust"])
            .agg(F.count(F.lit(1)).alias("n_acct"),
                 F.sum(F.when(F.col(D["dep_close"]).isNotNull(), 1).otherwise(0)).alias("n_closed"))
            .withColumn("shape",
                F.when(F.col("n_closed") == 0, "none_closed")
                 .when(F.col("n_closed") == F.col("n_acct"), "all_closed")
                 .otherwise("PARTIAL_CLOSURE"))
            .groupBy("shape").count().toPandas())
        save(partial, "09_partial_closure_shape",
             "*** PARTIAL_CLOSURE is the population where account-level and customer-level labels disagree ***")

---
## Stage 10 — Same-ledger test  *(Q8)*

**The question that reframes the whole study.** If balance change is just the
sum of these transactions, then deposit features and payment features are two
views of one table rather than two independent sources — and the only genuinely
new axis the graph contributes is *counterparty identity*, not amount or timing.

That is still a real contribution, but it changes the hypothesis and it predicts
which metrics can possibly add lift: the ones keyed on **who**, not the ones
keyed on **how much**.

Method: for a sample of accounts, aggregate signed transaction amounts per
account-day and compare to the day-over-day balance delta.

In [ ]:
STAGE = 10
# Signed amount. If the sign convention is already correct, use it as-is; if the
# amount is unsigned and direction carries the sign, apply direction here.
SIGN_FROM_DIRECTION = True   # <-- set from the Stage 4 result
DEBIT_VALUES = ["D", "DR", "DEBIT", "-1"]   # <-- set from the Stage 4 result

if SIGN_FROM_DIRECTION and C["direction"]:
    signed = F.when(F.upper(F.col(C["direction"]).cast("string")).isin(DEBIT_VALUES),
                    -F.abs(F.col(C["amount"]))).otherwise(F.abs(F.col(C["amount"])))
else:
    signed = F.col(C["amount"])

sample_accts = (dep.select(F.col(D["dep_acct"]).alias("acct")).distinct()
                   .limit(CONFIG["RECON_SAMPLE_ACCTS"]).cache())

txn_daily = (txn_w.join(F.broadcast(sample_accts), F.col(C["acct_id"]) == F.col("acct"))
    .groupBy(F.col("acct"), F.col(C["txn_date"]).alias("d"))
    .agg(F.sum(signed).alias("net_txn"), F.count(F.lit(1)).alias("n_txn")))

wspec = W.partitionBy("acct").orderBy("d")
bal_daily = (dep.join(F.broadcast(sample_accts), F.col(D["dep_acct"]) == F.col("acct"))
    .select(F.col("acct"), F.col(D["dep_date"]).alias("d"), F.col(D["dep_bal"]).alias("bal"))
    .withColumn("bal_prev", F.lag("bal").over(wspec))
    .withColumn("bal_delta", F.col("bal") - F.col("bal_prev"))
    .filter(F.col("bal_prev").isNotNull()))

recon = (bal_daily.join(txn_daily, ["acct", "d"], "left")
    .fillna({"net_txn": 0.0, "n_txn": 0})
    .withColumn("resid", F.col("bal_delta") - F.col("net_txn"))
    .withColumn("scale", F.greatest(F.abs(F.col("bal_delta")), F.abs(F.col("net_txn")), F.lit(1.0)))
    .withColumn("rel_resid", F.abs(F.col("resid")) / F.col("scale"))
    .withColumn("reconciles", F.col("rel_resid") <= CONFIG["RECON_TOL_REL"]))

summary = recon.agg(
    F.count(F.lit(1)).alias("n_account_days"),
    F.mean(F.col("reconciles").cast("double")).alias("share_reconciling"),
    F.expr("percentile_approx(rel_resid, 0.5)").alias("p50_rel_resid"),
    F.expr("percentile_approx(rel_resid, 0.9)").alias("p90_rel_resid"),
    F.corr("bal_delta", "net_txn").alias("corr_delta_vs_txn"),
).toPandas()
save(summary, "10_ledger_reconciliation",
     "*** Q8. share_reconciling near 1.0 = ONE ledger. Near 0 = two systems. ***")
_RESULTS["share_reconciling"] = float(summary["share_reconciling"].iloc[0])

In [ ]:
# If reconciliation is partial, the residual tells you what is missing — interest,
# fees, internal transfers, or a rail that is in the deposit system but not in
# staging. Profile the residual against the account-days with no transactions.
STAGE = 10
resid_prof = (recon.withColumn("bucket",
        F.when(F.col("n_txn") == 0, "no_txn_but_balance_moved")
         .when(F.col("reconciles"), "reconciles")
         .when(F.col("resid") > 0, "balance_rose_more_than_txn")
         .otherwise("balance_fell_more_than_txn"))
    .groupBy("bucket")
    .agg(F.count(F.lit(1)).alias("n"),
         F.expr("percentile_approx(abs(resid), 0.5)").alias("p50_abs_resid"))
    .toPandas())
resid_prof["share"] = resid_prof["n"] / resid_prof["n"].sum()
save(resid_prof, "10_reconciliation_residual_buckets",
     "no_txn_but_balance_moved = a money movement the staging table does not carry")

---
## Stage 11 — Join coverage: deposit book ↔ staging table

The coverage figure that made the prior study's verdict conditional. Under
deposit-anchored extraction this should be near-total rather than ~12% — but
"should be" is not a measurement, and if it is not near-total the reason needs
to be found before anything is built on it.

In [ ]:
STAGE = 11
dep_accts = dep.select(F.col(D["dep_acct"]).alias("k")).distinct()
txn_accts = txn_w.select(F.col(C["acct_id"]).alias("k")).distinct()

cov_j = (dep_accts.join(txn_accts.withColumn("_in_txn", F.lit(1)), "k", "left")
    .agg(F.count(F.lit(1)).alias("n_deposit_accounts"),
         F.sum(F.coalesce(F.col("_in_txn"), F.lit(0))).alias("n_with_transactions"))
    .toPandas())
cov_j["coverage"] = cov_j["n_with_transactions"] / cov_j["n_deposit_accounts"]
save(cov_j, "11_deposit_to_txn_coverage", "*** the number that replaces the ~12% figure ***")
_RESULTS["deposit_txn_coverage"] = float(cov_j["coverage"].iloc[0])

# Reverse direction: staging accounts with no deposit row. These are either
# non-deposit products or a join-key problem, and the two look identical.
rev = (txn_accts.join(dep_accts.withColumn("_in_dep", F.lit(1)), "k", "left")
    .agg(F.count(F.lit(1)).alias("n_txn_accounts"),
         F.sum(F.coalesce(F.col("_in_dep"), F.lit(0))).alias("n_with_deposits")).toPandas())
rev["coverage"] = rev["n_with_deposits"] / rev["n_txn_accounts"]
save(rev, "11_txn_to_deposit_coverage")

# Dtype trap check — the 2026-07 incident. If both sides are strings this is a
# no-op; if one is int64 the join above silently returned zero and looked plausible.
print("\ndeposit acct dtype:", dict(dep.dtypes)[D["dep_acct"]],
      "| txn acct dtype:", dict(txn_w.dtypes)[C["acct_id"]])

---
## Stage 12 — On-us self-payment ground truth

The calibration set for the self-payment matcher. Two PNC accounts belonging to
the same customer, moving money between themselves: same entity, same
name-string problem, **known answer**.

Exact-match precision and recall are measured here first. The off-us matcher
then inherits a calibrated threshold rather than an assumed one — with the
stated distribution shift that off-us names come from rail fields rather than
MDM and are dirtier.

Also produces the **base rate**, which matters more than the matcher: most
corporates permanently maintain multiple bank relationships, so a same-name
outflow is not attrition. The signal is a *new* destination or a step change in
share, not the level.

In [ ]:
STAGE = 12
onus = keyed.filter(F.col("_rtn_clean").isin(CONFIG["PNC_RTNS"]) & F.col("_has_acct"))

if D["dep_cust"]:
    acct2cust = dep.select(F.col(D["dep_acct"]).alias("acct"),
                           F.col(D["dep_cust"]).alias("cust")).distinct()

    truth = (onus
        .join(acct2cust.withColumnRenamed("acct", "src_acct").withColumnRenamed("cust", "src_cust"),
              F.col(C["acct_id"]) == F.col("src_acct"))
        .join(acct2cust.withColumnRenamed("acct", "dst_acct").withColumnRenamed("cust", "dst_cust"),
              F.col("_acct_clean") == norm_acct(F.col("dst_acct")))
        .withColumn("truth_self", F.col("src_cust") == F.col("dst_cust")))

    save(truth.groupBy("truth_self")
              .agg(F.count(F.lit(1)).alias("n"),
                   F.countDistinct("src_cust").alias("n_customers"),
                   F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"))
              .toPandas(),
         "12_onus_self_payment_truth",
         "the calibration set: same-customer transfers between two PNC accounts")

    # Confusion matrix for the v1 matcher (exact match on normalised names).
    # Needs the customer's own name, so it only runs with CUST_TABLE set.
    if C["cpty_name"] and CONFIG["CUST_TABLE"]:
        cust_dim = spark.table(CONFIG["CUST_TABLE"])
        own_name_col = resolve(cust_dim, "cpty_name", required=False) or "customer_name"
        own = (cust_dim.select(F.col(resolve(cust_dim, "cust_id")).alias("ocid"),
                               norm_name(F.col(own_name_col)).alias("own_name"))
                       .filter(F.col("own_name") != "").distinct())

        scored = (truth.join(own, F.col("src_cust") == F.col("ocid"), "inner")
                       .withColumn("pred_self",
                                   norm_name(F.col(C["cpty_name"])) == F.col("own_name")))

        cm = (scored.groupBy("truth_self", "pred_self")
                    .agg(F.count(F.lit(1)).alias("n"),
                         F.sum(F.abs(F.col(C["amount"]))).alias("abs_dollars"))
                    .toPandas())
        save(cm, "12_exact_match_confusion",
             "*** precision/recall of exact-name matching, on known answers ***")

        tp = cm.query("truth_self and pred_self")["n"].sum()
        fp = cm.query("not truth_self and pred_self")["n"].sum()
        fn = cm.query("truth_self and not pred_self")["n"].sum()
        pr = tp / (tp + fp) if (tp + fp) else float("nan")
        rc = tp / (tp + fn) if (tp + fn) else float("nan")
        save(pd.DataFrame([{"precision": pr, "recall": rc, "tp": tp, "fp": fp, "fn": fn}]),
             "12_exact_match_scores",
             "recall is the headroom fuzzy matching would buy; precision is what it would cost")

In [ ]:
# Base rate of same-name outflow, on-us and off-us. If this is high, the flag as
# specified in the brief fires on a large share of the book and is not a signal.
STAGE = 12
if C["cpty_name"] and C["cust_id"]:
    ego = keyed.filter(F.col("_has_name"))
    # Customer's own name — from the customer dim if available, else skip.
    if CONFIG["CUST_TABLE"]:
        cust_dim = spark.table(CONFIG["CUST_TABLE"])
        cn = resolve(cust_dim, "cpty_name", required=False) or "customer_name"
        own = cust_dim.select(F.col(resolve(cust_dim, "cust_id")).alias("cid"),
                              norm_name(F.col(cn)).alias("own_name")).distinct()
        ego = (ego.join(own, F.col(C["cust_id"]) == F.col("cid"), "left")
                  .withColumn("same_name", norm_name(F.col(C["cpty_name"])) == F.col("own_name")))

        base = (ego.withColumn("month", F.date_format(F.col(C["txn_date"]), "yyyy-MM"))
            .groupBy("month")
            .agg(F.countDistinct(F.when(F.col("same_name"), F.col(C["cust_id"]))).alias("n_cust_same_name"),
                 F.countDistinct(F.col(C["cust_id"])).alias("n_cust_total"),
                 (F.sum(F.when(F.col("same_name"), F.abs(F.col(C["amount"]))).otherwise(0.0))
                  / F.sum(F.abs(F.col(C["amount"])))).alias("same_name_dollar_share"))
            .withColumn("base_rate", F.col("n_cust_same_name") / F.col("n_cust_total"))
            .orderBy("month").toPandas())
        save(base, "12_same_name_base_rate",
             "*** if base_rate is high, measure the DELTA not the LEVEL ***")
    else:
        print("CUST_TABLE not set — cannot compute the same-name base rate. Set it and re-run Stage 12.")

---
## Stage 13 — Episode sizing  *(Q13)*

The power calculation. Applies the current 30% rule to the deposit panel, then
walks the brief's §6.2 exclusion funnel and reports the surviving episode count
at each step.

**This decides whether Steps 3–6 of the brief are worth running.** Twenty-five
metrics against matched controls at two horizons needs a few thousand episodes,
not a few hundred.

In [ ]:
STAGE = 13
# Monthly balance per customer. Roll-up rule: SUM across accounts, which treats a
# customer as one relationship. Stage 9's partial-closure count is how wrong that is.
bal_m = (dep
    .withColumn("month", F.trunc(F.col(D["dep_date"]), "MM"))
    .groupBy(F.col(D["dep_cust"]).alias("cust") if D["dep_cust"] else F.col(D["dep_acct"]).alias("cust"),
             "month")
    .agg(F.mean(F.col(D["dep_bal"])).alias("bal")))   # mean-of-daily; swap to last() for month-end

w9 = W.partitionBy("cust").orderBy("month")
ep = (bal_m
    .withColumn("avg3",  F.avg("bal").over(w9.rowsBetween(-2, 0)))
    .withColumn("avg6p", F.avg("bal").over(w9.rowsBetween(-8, -3)))
    .withColumn("n_hist", F.count("bal").over(w9.rowsBetween(-11, 0)))
    .withColumn("flag", (F.col("avg3") <= (1 - CONFIG["CURRENT_RULE_DROP"]) * F.col("avg6p"))))
# NOTE: percentile_approx is not a window function in Spark. The trailing-12 median
# that §6.1 of the brief needs for T0 back-tracing is a separate pass (collect_list +
# an array median, or a self-join) — deliberately out of scope here. This stage only
# sizes the population under the CURRENT rule.

funnel = []
def step(df, label):
    n_ep = df.filter(F.col("flag")).select("cust").distinct().count()
    funnel.append({"step": label, "n_customers_flagged": n_ep})
    return df

step(ep, "1_raw_30pct_rule")
ep2 = ep.filter(F.col("n_hist") >= 12);                       step(ep2, "2_plus_12mo_burnin")
ep3 = ep2.filter(F.col("avg6p") >= CONFIG["BALANCE_FLOOR"]);  step(ep3, "3_plus_balance_floor")

# Persistence: the depressed level holds three months.
ep4 = (ep3.withColumn("fwd3", F.avg("bal").over(w9.rowsBetween(1, 3)))
          .filter((~F.col("flag")) | (F.col("fwd3") <= 1.1 * F.col("avg3"))))
step(ep4, "4_plus_persistence_3mo")

save(pd.DataFrame(funnel), "13_episode_funnel",
     "*** Q13. The last row is the sample size Steps 3-6 actually have. ***")

In [ ]:
# Graph-visible subset of the surviving episodes — the number that was missing
# from the brief entirely.
STAGE = 13
survivors = ep4.filter(F.col("flag")).select("cust").distinct()
gv = (survivors.join(txn_w.select(F.col(C["cust_id"] or C["acct_id"]).alias("cust")).distinct()
                     .withColumn("_seen", F.lit(1)), "cust", "left")
      .agg(F.count(F.lit(1)).alias("n_episodes"),
           F.sum(F.coalesce(F.col("_seen"), F.lit(0))).alias("n_graph_visible")).toPandas())
gv["coverage"] = gv["n_graph_visible"] / gv["n_episodes"]
save(gv, "13_episodes_graph_visible", "*** the population any payment metric can actually be tested on ***")
_RESULTS["episodes_graph_visible"] = int(gv["n_graph_visible"].iloc[0])

---
## Stage 14 — Summary

Assembles the headline numbers into one row. This is what goes into the reply to
the brief — every claim in that reply should trace to a cell here or to one of
the CSVs above.

In [ ]:
STAGE = 14
summary_rows = [
    ("Q1  staging rows",                     _RESULTS.get("shape", {}).get("n_rows")),
    ("Q1  date range",                       f"{_RESULTS.get('shape', {}).get('min_txn_date')} → "
                                             f"{_RESULTS.get('shape', {}).get('max_txn_date')}"),
    ("Q3  cpty name coverage, $-weighted",   _RESULTS.get("name_rate_dollars_overall")),
    ("Q5  hard-key coverage, $-weighted",    _RESULTS.get("hardkey_rate_dollars_overall")),
    ("Q8  share of account-days reconciling", _RESULTS.get("share_reconciling")),
    ("Q9  closures observed in window",      _RESULTS.get("closures_in_window")),
    ("—   deposit→txn account coverage",     _RESULTS.get("deposit_txn_coverage")),
    ("Q13 graph-visible episodes",           _RESULTS.get("episodes_graph_visible")),
]
save(pd.DataFrame(summary_rows, columns=["question", "value"]), "14_SUMMARY")

print("""
READING THE SUMMARY
-------------------
Q3 name coverage, dollar-weighted
    > 0.70  Group A is live as specified.
    0.40–0.70  Group A is live on a biased subset; every figure needs the rail split alongside it.
    < 0.40  Group A is a rail-specific metric, not a book-wide one. Say so in the reply.

Q8 share reconciling
    > 0.95  ONE ledger. Deposit and payment features are two views of one table.
            Reframe the hypothesis: counterparty IDENTITY is the new axis, not amount or timing.
    < 0.50  Two systems. The original independent-sources framing holds.
    in between  Find what is in the residual (Stage 10 buckets) before deciding.

Q9 closures
    Any populated, dated close field replaces the balance-derived label and removes
    the circularity. This is worth more than any metric in §7 of the brief.

Q13 graph-visible episodes
    < 300   Steps 3-6 are underpowered. Report Steps 1-2 and stop.
    300-1000  Descriptive separation curves only; no scorecard, no model.
    > 1000  The brief's plan is executable as written.
""")